<a href="https://colab.research.google.com/github/sdhabalia24/CS482-Project/blob/main/CS482_Milestone3_SaahilDhabalia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Part 1: The Movie Database API

This Colab Notebook is focused on collecting the data from The Movie Database API. Running it can take several minutes as the data has to be processed. For that reason, the collection done via multi-threading. Once completed, a csv file is downloaded, and that is the file we base the Mielstone 3 project on.

In [73]:
import os
import requests
import random
import time
import json
from google.colab import userdata
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading
from tqdm.auto import tqdm

os.environ["TMDB_API_KEY"] = userdata.get("TMDB_API_KEY")
TMDB_API_KEY = os.environ["TMDB_API_KEY"]  # TMBD API Key stored in environment variable

BASE_URL = "https://api.themoviedb.org/3"

thread_local = threading.local()
def get_session():
  if not hasattr(thread_local, "session"):
    thread_local.session = requests.Session()
    thread_local.session.params = {"api_key": TMDB_API_KEY}
  return thread_local.session

In [74]:
def api_request(endpoint, params=None):

    session = get_session()

    for attempt in range(5):

        try:
            response = session.get(
                f"{BASE_URL}/{endpoint}",
                params=params,
                timeout=30
            )

            if response.status_code == 429:
                delay = int(
                    response.headers.get("Retry-After", 5)
                )
                time.sleep(delay)
                continue

            if response.status_code == 404:
                return None

            if response.status_code >= 500:
                time.sleep(2 ** attempt)
                continue

            response.raise_for_status()

            return response.json()

        except (requests.ConnectionError, requests.Timeout):
            time.sleep(2 ** attempt)

    raise RuntimeError(f"Request failed: {endpoint}")

In [75]:
def discover_movies(start_year, end_year, pages_per_year):
  movie_ids = set()
  for year in range(start_year, end_year + 1):
    params = {
        "primary_release_year": year,
        "with_original_language": "en",
        "sort_by": "popularity.desc",
        "vote_count.gte": 20,
        "include_adult": False,
        "include_video": False
    }

    initial = api_request("discover/movie", params)
    total_pages = min(initial["total_pages"], 500)
    selected_pages = random.sample(
        range(1, total_pages + 1),
        min(pages_per_year, total_pages)
    )

    for page in selected_pages:
      params["page"] = page
      data = api_request("discover/movie", params)
      for movie in data["results"]:
        movie_ids.add(movie["id"])
    print(f"Year {year} completed. Total: {len(movie_ids)}")
  return sorted(movie_ids)


In [76]:
movie_ids = discover_movies(1950, 2026, 100)
print(f"Collected {len(movie_ids)} unique movies")

Year 1950 completed. Total: 170
Year 1951 completed. Total: 332
Year 1952 completed. Total: 496
Year 1953 completed. Total: 668
Year 1954 completed. Total: 841
Year 1955 completed. Total: 1016
Year 1956 completed. Total: 1178
Year 1957 completed. Total: 1369
Year 1958 completed. Total: 1527
Year 1959 completed. Total: 1677
Year 1960 completed. Total: 1810
Year 1961 completed. Total: 1930
Year 1962 completed. Total: 2065
Year 1963 completed. Total: 2193
Year 1964 completed. Total: 2345
Year 1965 completed. Total: 2499
Year 1966 completed. Total: 2667
Year 1967 completed. Total: 2846
Year 1968 completed. Total: 2999
Year 1969 completed. Total: 3165
Year 1970 completed. Total: 3336
Year 1971 completed. Total: 3544
Year 1972 completed. Total: 3746
Year 1973 completed. Total: 3940
Year 1974 completed. Total: 4138
Year 1975 completed. Total: 4280
Year 1976 completed. Total: 4450
Year 1977 completed. Total: 4637
Year 1978 completed. Total: 4813
Year 1979 completed. Total: 5006
Year 1980 compl

In [77]:
# Save IDs
with open("movie_ids.json", "w") as f:
  json.dump(movie_ids, f)

In [78]:
data = api_request(
    "movie/550",
    {"append_to_response": "credits"}
)

params = {
    "append_to_resoonse": "credits,keywords,release_dates"
}

In [79]:
def get_movie_details(movie_id):
  data = api_request(
      f"movie/{movie_id}",
      {"append_to_response": "credits,keywords,release_dates"}
  )

  if data is None:
    return None

  cast = data["credits"]["cast"]
  crew = data["credits"]["crew"]
  directors = [
      person["name"]
      for person in crew
      if person["job"] == "Director"
  ]

  actors = [
      person["name"]
      for person in cast[:5] # top 5
  ]

  return {
      "id": data["id"],
      "title": data["title"],
      "budget": data["budget"],
      "revenue": data["revenue"],
      "runtime": data["runtime"],
      "release_date": data["release_date"],
      "original_language": data["original_language"],
      "popularity": data["popularity"],
      "vote_average": data["vote_average"],
      "vote_count": data["vote_count"],
      "overview": data["overview"],
      "genres": [
          g["name"] for g in data["genres"]
      ],
      "production_companies": [
          c["name"] for c in data["production_companies"]
      ],
      "production_countries": [
          c["name"] for c in data["production_countries"]
      ],
      "directors": directors,
      "actors": actors,
      "cast_size": len(cast),
      "crew_size": len(crew),
      "keywords": [
          k["name"]
          for k in data["keywords"]["keywords"]
      ]
  }

In [80]:
MAX_WORKERS = 5

with open("movies_raw.jsonl", "a+") as file:

    file.seek(0)

    # Identify previously collected movies
    completed = {
        json.loads(line)["id"]
        for line in file
        if line.strip()
    }

    # Only retrieve new movies
    remaining_ids = [
        movie_id for movie_id in movie_ids
        if movie_id not in completed
    ]

    print(f"Already collected: {len(completed)}")
    print(f"Remaining movies: {len(remaining_ids)}")

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:

        futures = {
            executor.submit(get_movie_details, movie_id): movie_id
            for movie_id in remaining_ids
        }

        for future in tqdm(
            as_completed(futures),
            total=len(futures),
            desc="Collecting movies"
        ):

            try:
                movie = future.result()

                if movie is not None:
                    file.write(json.dumps(movie) + "\n")
                    file.flush()

            except Exception as e:
                movie_id = futures[future]
                print(f"Error retrieving {movie_id}: {e}")

Already collected: 10783
Remaining movies: 25186


In [92]:
import pandas as pd
from google.colab import files

df = pd.read_json("movies_raw.jsonl", lines=True)
#df = df.drop_duplicates(subset="id")
df.to_csv("movies_raw.csv", index=False)
files.download("movies_raw.csv")
print(df.shape)
df.head(10)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

(35969, 19)


,id,title,budget,revenue,runtime,release_date,original_language,popularity,vote_average,vote_count,overview,genres,production_companies,production_countries,directors,actors,cast_size,crew_size,keywords
0,13,Forrest Gump,55000000,677387716,142,1994-06-23,en,32.6376,8.462,30966,A man with a low IQ has accomplished great thi...,"[Comedy, Drama, Romance]","[Paramount Pictures, The Steve Tisch Company, ...",[United States],[Robert Zemeckis],"[Tom Hanks, Robin Wright, Gary Sinise, Mykelti...",144,178,"[new year's eve, vietnam war, vietnam veteran,..."
1,6,Judgment Night,21000000,12136938,110,1993-10-15,en,2.2585,6.455,397,"Four young friends, while taking a shortcut en...","[Action, Crime, Drama, Thriller]","[Largo Entertainment, JVC]","[United States, Japan]",[Stephen Hopkins],"[Emilio Estevez, Cuba Gooding Jr., Denis Leary...",31,19,"[drug dealer, chicago, illinois, escape, one n..."
2,11,Star Wars,11000000,775398007,121,1977-05-25,en,30.8131,8.206,23222,Princess Leia is captured and held hostage by ...,"[Adventure, Action, Science Fiction]",[Lucasfilm Ltd.],[United States],[George Lucas],"[Mark Hamill, Harrison Ford, Carrie Fisher, Pe...",105,95,"[empire, galaxy, rebellion, android, hermit, s..."
3,14,American Beauty,15000000,356296601,122,1999-09-15,en,11.0447,7.996,13585,"Lester Burnham, a depressed suburban father in...",[Drama],"[DreamWorks Pictures, Jinks/Cohen Company]",[United States],[Sam Mendes],"[Kevin Spacey, Annette Bening, Thora Birch, We...",41,148,"[estate agent, adultery, depression, first tim..."
4,25,Jarhead,72000000,97076152,123,2005-11-04,en,5.2985,6.695,3346,Jarhead is a film about a US Marine Anthony Sw...,"[Drama, War]","[Red Wagon Entertainment, Neal Street Producti...","[United Kingdom, United States]",[Sam Mendes],"[Jake Gyllenhaal, Jamie Foxx, Peter Sarsgaard,...",32,171,"[sniper, california, gulf war, petrol, marine ..."
5,66,Absolute Power,50000000,50100000,121,1997-02-14,en,4.5567,6.663,1334,A master thief coincidentally is robbing a hou...,"[Crime, Drama, Action]","[Castle Rock Entertainment, Malpaso Productions]",[United States],[Clint Eastwood],"[Clint Eastwood, Gene Hackman, Ed Harris, Laur...",24,63,"[daughter, rape, assassination, usa president,..."
6,69,Walk the Line,28000000,186800000,136,2005-09-13,en,6.6994,7.500,3273,A chronicle of country music legend Johnny Cas...,"[Drama, Music, Romance]","[Mars Media Beteiligungs, Tree Line Films, Cat...","[Germany, United States]",[James Mangold],"[Joaquin Phoenix, Reese Witherspoon, Ginnifer ...",89,69,"[prison, adultery, concert, rock 'n' roll, mus..."
7,73,American History X,20000000,23900000,119,1998-07-01,en,12.8536,8.324,13173,Derek Vineyard is paroled after serving 3 year...,[Drama],"[New Line Cinema, Savoy Pictures, The Turman-M...",[United States],[Tony Kaye],"[Edward Norton, Edward Furlong, Beverly D'Ange...",50,143,"[prison, neo-nazism, rape, skinhead, sibling r..."
8,58,Pirates of the Caribbean: Dead Man's Chest,200000000,1066179747,151,2006-07-06,en,21.8366,7.394,17889,Captain Jack Sparrow’s got a blood debt to pay...,"[Adventure, Fantasy, Action]","[Walt Disney Pictures, Jerry Bruckheimer Films...",[United States],[Gore Verbinski],"[Johnny Depp, Orlando Bloom, Keira Knightley, ...",72,306,"[witch, daughter, ship, exotic island, monster..."
9,85,Raiders of the Lost Ark,18000000,389925971,115,1981-06-12,en,15.9154,7.928,14263,When Dr. Indiana Jones – the tweed-suited prof...,"[Adventure, Action]","[Paramount Pictures, Lucasfilm Ltd.]",[United States],[Steven Spielberg],"[Harrison Ford, Karen Allen, Paul Freeman, Joh...",50,195,"[egypt, treasure, medallion, swastika, saving ..."


In [90]:
# Determine Missing Data
print("Total movies:", len(df))

print("Missing revenue:", (df["revenue"] == 0).sum())
print("Missing budget:", (df["budget"] == 0).sum())

print("Missing revenue percentage:",
      (df["revenue"] == 0).mean() * 100)

print("Missing budget percentage:",
      (df["budget"] == 0).mean() * 100)

Total movies: 35969
Missing revenue: 24384
Missing budget: 24150
Missing revenue percentage: 67.79170952764882
Missing budget percentage: 67.14114932302816


In [91]:
print("Discovered movie IDs:", len(movie_ids))
print("Unique movie IDs:", len(set(movie_ids)))

df_json = pd.read_json("movies_raw.jsonl", lines=True)
df_csv = pd.read_csv("movies_raw.csv")

print("Movies in JSONL:", len(df_json))
print("Movies in CSV:", len(df_csv))
print("Movies in current DataFrame:", len(df))

Discovered movie IDs: 35969
Unique movie IDs: 35969
Movies in JSONL: 35969
Movies in CSV: 35969
Movies in current DataFrame: 35969
